# C4-classical-ml-practice — Session 2: k-Nearest Neighbors and Feature Scaling

*One class session, roughly 85 minutes. Prerequisites: Session 1
(the table → `(X, y)` bridge), C1-ml-fundamentals (supervised
classification, train/test discipline, overfitting, accuracy),
F2-vectors (Euclidean and Manhattan distance), F5-probability
(variance), and F1's broadcasting.*

**This session:** your first real classifier, built from parts you
already own.
k-nearest neighbors is F2's distances plus a vote: to classify a new
point, find the $k$ closest training rows and let their labels vote.
We build it **from first principles** — distances, `argsort`, majority
vote — before touching any library, watch what the knob $k$ does, then
meet the method's Achilles' heel: features on different scales silently
rig the vote (a fully worked distortion example), and standardization —
F5's variance put to work — un-rigs it.
Only then does sklearn enter, as a *convenience wrapper around
computations you have already implemented and can verify*.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260804

## 1. Classify by Similarity

**Motivation.**
C1 defined supervised classification: learn a rule mapping a feature
row to a class label from labeled examples.
The simplest possible rule needs no equations at all:
**a new point probably has the same label as the training points it
sits closest to.**
That single sentence — the *similarity assumption* — is the entire
theory of k-nearest neighbors.
Everything else is bookkeeping: what "closest" means (F2 distances) and
how many neighbors get a say (the knob $k$).

**The setup.**
As always: `X_train` of shape `(n, d)` — one row per labeled example —
and `y_train` of shape `(n,)`.
A **query** is any new row of $d$ features we must label.
Note what is *absent*: no training phase, no fitted coefficients.
kNN just *remembers* the training set; all work happens at prediction
time.

Two seeded blobs make the geometry visible:

In [ ]:
rng = np.random.default_rng(SEED)
X_train = np.vstack([rng.normal((0.0, 0.0), 1.0, (25, 2)),    # class 0
                     rng.normal((3.0, 2.0), 1.0, (25, 2))])   # class 1
y_train = np.array([0] * 25 + [1] * 25)

plt.figure(figsize=(5.5, 4))
for cls, marker in [(0, "o"), (1, "s")]:
    pts = X_train[y_train == cls]
    plt.scatter(pts[:, 0], pts[:, 1], marker=marker, label=f"class {cls}")
query = np.array([1.6, 1.2])
plt.scatter(*query, marker="*", s=220, color="black", label="query ?")
plt.legend(); plt.grid(linewidth=0.3)
plt.title("Which class is the starred point?")
plt.show()

### Checkpoint 1

1. In one sentence, state the similarity assumption that kNN relies
   on — and give one real dataset where it sounds reasonable and one
   where it sounds dubious.
2. kNN has no `fit` computation to speak of.
   What does it store instead, and when does the actual work happen?
3. What are the shapes of everything involved in classifying 7 queries
   against 50 training rows with 2 features?

## 2. Distances Between Rows (F2, Recapped)

**Definition** (from F2, now applied to feature rows).
For rows $u, v$ with $d$ features:

$$\text{Euclidean: } \; \operatorname{dist}(u, v) = \sqrt{\sum_{j} (u_j - v_j)^2}
\qquad
\text{Manhattan: } \; \sum_{j} |u_j - v_j|.$$

For *ranking* neighbors the square root is optional — it is increasing,
so sorting by squared distance gives the same order and saves the
`sqrt`.
This unit ranks by **squared Euclidean distance** unless stated
otherwise.

**All distances at once** (F1 broadcasting, no loops):
`X_train - query` broadcasts the query across all rows; square, sum
along `axis=1`, done.
For *many* queries, insert axes so every (query, training-row) pair
gets its own slot — the same alignment trick you used in F1:

In [ ]:
def dist2_to_all(X, q):
    # squared Euclidean distance from one query q (d,) to every row of X (n, d)
    return ((X - q) ** 2).sum(axis=1)                    # (n,)


def dist2_matrix(X, Q):
    # all pairs: rows of Q (m, d) vs rows of X (n, d)  ->  (m, n)
    return ((Q[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)


d2 = dist2_to_all(X_train, query)
print("distances to all 50 training rows:", d2.shape)
print("closest three squared distances:", np.sort(d2)[:3].round(3))

Q = np.array([[1.6, 1.2], [-1.0, 0.0], [4.0, 3.0]])
D2 = dist2_matrix(X_train, Q)
print("3 queries x 50 rows:", D2.shape)
print("agreement with single-query version:", np.abs(D2[0] - d2).max())

### Checkpoint 2

1. By hand: squared Euclidean and Manhattan distances between rows
   $(2, 1, 0)$ and $(0, 2, 2)$.
2. Why is ranking by squared distance guaranteed to produce the same
   neighbor order as ranking by true distance?
3. In `dist2_matrix`, state the shape after `(Q[:, None, :] - X[None, :, :])`
   and what its `[i, j, :]` slice holds.

## 3. kNN From First Principles

**The algorithm** — all of it:

1. Compute the query's distance to every training row.
2. Take the indices of the $k$ smallest (`np.argsort(d2)[:k]`).
3. Collect those rows' labels and take the **majority vote**.

**Worked example (by hand).**
Training set, labels 0/1, query $q = (4, 4)$:

| row | point | label | squared dist to $q$ |
|---|---|---|---|
| 0 | (1, 1) | 0 | $9 + 9 = 18$ |
| 1 | (2, 2) | 0 | $4 + 4 = 8$ |
| 2 | (3, 3) | 0 | $1 + 1 = 2$ |
| 3 | (6, 5) | 1 | $4 + 1 = 5$ |
| 4 | (7, 7) | 1 | $9 + 9 = 18$ |

$k = 1$: nearest is row 2 (dist² 2, label 0) → predict **0**.
$k = 3$: rows 2, 3, 1 (dist² 2, 5, 8; labels 0, 1, 0) → vote 2–1 →
predict **0**.
Note $k = 1$ and $k = 3$ *can* disagree — here they happen not to.

**Implementation.**
With 0/1 labels a majority vote is just a mean: the $k$ neighbor labels
average above $\tfrac12$ exactly when 1s outnumber 0s (use odd $k$ so
ties cannot happen).

In [ ]:
def knn_predict(X_tr, y_tr, X_q, k):
    # Labels for each row of X_q by majority vote among the k nearest
    # rows of X_tr.  Binary 0/1 labels; use odd k.
    d2 = dist2_matrix(X_tr, X_q)                 # (m, n)
    nearest = np.argsort(d2, axis=1)[:, :k]      # (m, k) indices into X_tr
    votes = y_tr[nearest]                        # (m, k) neighbor labels
    return (votes.mean(axis=1) > 0.5).astype(int)


X_toy = np.array([[1., 1.], [2., 2.], [3., 3.], [6., 5.], [7., 7.]])
y_toy = np.array([0, 0, 0, 1, 1])
q44 = np.array([[4., 4.]])
print("k=1:", knn_predict(X_toy, y_toy, q44, 1))   # [0]
print("k=3:", knn_predict(X_toy, y_toy, q44, 3))   # [0]

print("blob query:", knn_predict(X_train, y_train, query[None, :], 5))

### Checkpoint 3

1. By hand, for the 5-row table above: classify $q = (5, 5)$ with
   $k = 1$ and with $k = 3$.
   (Compute all five squared distances.)
2. In `knn_predict`, what exactly does `y_tr[nearest]` contain, and
   why does `votes.mean(axis=1) > 0.5` implement a majority vote for
   0/1 labels?
3. Modify the vote for **three** classes labeled 0/1/2: why does the
   mean trick break, and what would you use instead?
   (Hint: `np.unique(..., return_counts=True)` per query row.)

## 4. The Knob k: Memorizing versus Smoothing

**Motivation.**
$k$ is kNN's only real setting, and it retells C1's overfitting story
in miniature.

**The two extremes.**

- **$k = 1$** memorizes.
  Every training point's nearest neighbor is *itself*, so training
  accuracy is exactly 1.0 — by construction, not by merit.
  The decision boundary wraps around every stray point: noise gets its
  own little island.
- **$k = n$** averages *everything*: every query receives the
  training set's majority label.
  Maximal smoothing, zero attention to where the query actually is.

Between the extremes, growing $k$ smooths the boundary: single noisy
points get outvoted by their surroundings.
The *right* $k$ is a data question, answered honestly in Session 3 —
never by looking at training accuracy, which $k = 1$ maxes out for
free.

In [ ]:
for k in (1, 5, 25):
    train_preds = knn_predict(X_train, y_train, X_train, k)
    print(f"k={k:2d}: train accuracy = {(train_preds == y_train).mean():.3f}")

In [ ]:
# Decision regions: classify a whole grid of queries at once (no loops).
gx, gy = np.meshgrid(np.linspace(-3, 6, 160), np.linspace(-3, 5.5, 160))
grid = np.column_stack([gx.ravel(), gy.ravel()])          # (25600, 2) queries

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, k in zip(axes, (1, 5, 25)):
    Z = knn_predict(X_train, y_train, grid, k).reshape(gx.shape)
    ax.contourf(gx, gy, Z, levels=[-0.5, 0.5, 1.5], alpha=0.25)
    for cls, marker in [(0, "o"), (1, "s")]:
        pts = X_train[y_train == cls]
        ax.scatter(pts[:, 0], pts[:, 1], marker=marker, s=18)
    ax.set_title(f"k = {k}")
plt.tight_layout()
plt.show()

$k = 1$: ragged islands around individual points.
$k = 5$: a sane frontier.
$k = 25$ (half the data): the boundary barely notices local structure.

### Checkpoint 4

1. Explain *why* $k = 1$ training accuracy is always 1.0 (when all
   training points are distinct), and which C1 concept that
   illustrates.
2. A dataset has 40 points: 25 of class 0, 15 of class 1.
   What does every query get classified as when $k = 40$, and what is
   the training accuracy then?
3. Between $k = 1$ and $k = 25$, which prediction changes more if one
   single training point's label were flipped?
   Why?

## 5. The Scaling Distortion — a Worked Example

**Motivation.**
Distances add feature contributions *in the features' raw units*.
If one column lives on a scale a thousand times larger, it does not
just influence the distance — it **is** the distance, and every other
feature becomes decoration.
This is kNN's central practical failure mode, and the exam's applied
problem is built to punish anyone who forgets it.

**Worked example — watch a big-scale feature rig the vote.**
Two customers with known plans, one new customer to classify, features
= (height in meters, income in dollars per year):

| | height (m) | income (\$/yr) | plan |
|---|---|---|---|
| A | 1.80 | 30000 | plan-A |
| B | 1.60 | 31000 | plan-B |
| query $q$ | 1.78 | 30800 | ? |

By height, $q$ is practically A's twin ($|\Delta| = 0.02$ vs $0.18$ —
nine times closer).
Squared distances, by hand:

$$d^2(q, A) = (0.02)^2 + (800)^2 = 0.0004 + 640000 = 640000.0004,$$
$$d^2(q, B) = (0.18)^2 + (200)^2 = 0.0324 + 40000 = 40000.0324.$$

The 1-NN is **B** — decided *entirely* by income; height's contribution
(at most $0.0324$) could never compete with the income gaps in the
hundreds of thousands.
Whether that is right depends on which feature *should* matter, but
notice the units chose for you: measure height in millimeters instead
and the verdict flips back.
A classifier whose answer depends on the units of measurement is
answering the wrong question.

**The general statement.**
If feature $j$'s typical spread is $s_j$, its typical contribution to
$d^2$ is of order $s_j^2$: a feature with 1000× the spread carries
1 000 000× the weight.
Raw Euclidean kNN is *implicitly weighted* by squared feature scales —
weights chosen by fiat of the units, not by the data.

In [ ]:
A = np.array([1.80, 30000.0]);  B = np.array([1.60, 31000.0])
q = np.array([1.78, 30800.0])

d2A = ((q - A) ** 2).sum();  d2B = ((q - B) ** 2).sum()
print(f"d2(q, A) = {d2A:.4f}")
print(f"d2(q, B) = {d2B:.4f}   -> raw 1-NN is {'A' if d2A < d2B else 'B'}")

# Height's largest possible say vs income's actual say:
print("height contribution to d2(q, B):", (q[0] - B[0]) ** 2)
print("income contribution to d2(q, B):", (q[1] - B[1]) ** 2)

### Checkpoint 5

1. Recompute the worked example with height measured in
   **millimeters** (multiply the height column by 1000; income
   unchanged).
   Which point is the 1-NN now?
   Show both squared distances.
2. Features: `age` (spread ≈ 12 years) and `salary` (spread ≈ \$18000).
   Roughly what ratio of influence do they carry in squared Euclidean
   distance?
3. True or false, with one sentence: "multiplying *every* feature by
   10 changes which training point is a query's nearest neighbor."

## 6. Standardization: F5's Variance at Work

**Motivation.**
The fix must make "one typical spread" the common currency of every
column.
F5 already gave us the measure of spread: **variance**, the expected
squared deviation from the mean, and its square root $\sigma$, the
standard deviation.

**Definition (standardization / z-scoring).**
For feature column $j$ with mean $\mu_j$ and standard deviation
$\sigma_j = \sqrt{\operatorname{Var}(\text{column } j)}$:

$$z_{ij} \;=\; \frac{x_{ij} - \mu_j}{\sigma_j}.$$

Each value becomes "how many standard deviations from this column's
mean" — a pure, unitless number.
After standardizing, every column has mean $0$ and variance $1$
(F5: subtracting a constant leaves variance unchanged; dividing by
$\sigma$ divides the variance by $\sigma^2$), so each feature's typical
contribution to $d^2$ is the same order 1.

**The discipline that comes with it.**
$\mu_j$ and $\sigma_j$ are *estimated from the training rows only*, and
those same training statistics are applied to every later query or
test row.
Two reasons:

- a lone query is one row — it has no spread of its own to divide by;
- test rows must not influence any number the classifier uses, or the
  test stops measuring generalization (C1's discipline; Session 3
  returns to this as *leakage*).

**Worked example, concluded.**
Standardize the customer table with the training stats (the two rows A
and B) and re-run the vote:

In [ ]:
train = np.vstack([A, B])
mu = train.mean(axis=0)                        # [1.7, 30500.0]
sigma = train.std(axis=0)                      # [0.1,   500.0]  (population std, F5)
print("mu:", mu, " sigma:", sigma)

Az, Bz, qz = (A - mu) / sigma, (B - mu) / sigma, (q - mu) / sigma
print("A_z:", Az, " B_z:", Bz, " q_z:", qz.round(3))

d2Az = ((qz - Az) ** 2).sum();  d2Bz = ((qz - Bz) ** 2).sum()
print(f"d2(q, A) scaled = {d2Az:.2f}")                       # 2.60
print(f"d2(q, B) scaled = {d2Bz:.2f}   -> 1-NN is now {'A' if d2Az < d2Bz else 'B'}")

Scaled squared distances $2.60$ vs $3.40$: the 1-NN flips to **A**.
Height's nine-fold closeness finally counts; income still participates
(it keeps $q$ closer to B *in that coordinate*), but as one vote among
equals, not as a dictator.

### Checkpoint 6

1. Verify by hand that the standardized height column of the training
   table is exactly $(+1, -1)$, and state which F5 facts make the
   standardized column's mean 0 and variance 1 in general.
2. Why must the query be standardized with the *training* $\mu$ and
   $\sigma$ rather than statistics that include the query itself?
   Give both reasons from the text.
3. A feature column is constant across the training set.
   What goes wrong in the formula, and what does that suggest about
   the feature's usefulness?

## 7. Enter sklearn — a Wrapper You Can Verify

**Motivation.**
You now own every computation kNN needs.
**scikit-learn** (sklearn) packages them behind a uniform API used by
essentially all classical-ML code — and by the exam's applied problem,
whose rules (paraphrased) *allow scikit-learn but restrict the model
family to k-nearest neighbors*.
Fluency with this specific corner of sklearn is therefore exam craft,
not just convenience.

**The estimator API** — three verbs, shared by every sklearn model:

```python
model.fit(X_train, y_train)     # learn / memorize
model.predict(X_query)          # labels for new rows
model.score(X_test, y_test)     # accuracy on labeled data
```

**The two objects this unit needs.**

- `KNeighborsClassifier(n_neighbors=k)` — exactly Section 3's
  algorithm (it handles multi-class votes and distance ties
  internally).
- `StandardScaler()` — exactly Section 6's standardization:
  `fit(X_train)` records $\mu, \sigma$ per column;
  `transform(anything)` applies **those stored training statistics**.
  The fit/transform split *is* the train-statistics discipline, built
  into the API.

Never trust a wrapper you haven't tested against your own
implementation:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# 1) The classifier agrees with our first-principles version everywhere:
rng = np.random.default_rng(SEED)
probes = rng.normal(1.5, 2.0, (200, 2))              # random query points

sk_model = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
ours = knn_predict(X_train, y_train, probes, 5)
print("sklearn vs first-principles, 200 queries agree:",
      (sk_model.predict(probes) == ours).all())

# 2) The scaler agrees with the formula:
scaler = StandardScaler().fit(X_train)
manual = (X_train - X_train.mean(axis=0)) / X_train.std(axis=0)
print("scaler vs formula, max gap:",
      np.abs(scaler.transform(X_train) - manual).max())
print("scaler stored mean:", scaler.mean_.round(3))

**Worked exam-style example (the register, paraphrased).**

> **Task.**
> Arrays `X_all (60, 3)`, `y_all (60,)` are given (three sensor
> features on wildly different scales; labels 0 = ok, 1 = faulty).
> **(a)** Split off the last 20 rows (after a seeded shuffle) as a
> test set: `X_tr, y_tr, X_te, y_te`.
> **(b)** Fit `KNeighborsClassifier(n_neighbors=5)` on the raw
> training data; report `acc_raw` on the test set.
> **(c)** Standardize with a `StandardScaler` **fitted on the training
> rows only**; refit; report `acc_scaled`.
> **Constraints (zero points): models other than
> `KNeighborsClassifier`; fitting the scaler on any test row.**

**Solution, narrated.**
(a) is C1's seeded permutation split; (b) is three lines of estimator
API; (c) threads the *same* fitted scaler over both splits — fit on
train, transform both.

In [ ]:
# The given data: 60 machines, features (temp_C, vibration, pressure_Pa).
rng = np.random.default_rng(SEED)
ok = np.column_stack([rng.normal(65.0, 1.8, 30), rng.normal(3.0, 0.7, 30),
                      rng.normal(101000.0, 1400.0, 30)])
faulty = np.column_stack([rng.normal(68.2, 1.8, 30), rng.normal(4.6, 0.7, 30),
                          rng.normal(101000.0, 1400.0, 30)])
X_all = np.vstack([ok, faulty])
y_all = np.array([0] * 30 + [1] * 30)

# (a) seeded shuffle + split
perm = rng.permutation(60)
X_shuf, y_shuf = X_all[perm], y_all[perm]
X_tr, y_tr = X_shuf[:40], y_shuf[:40]
X_te, y_te = X_shuf[40:], y_shuf[40:]

# (b) raw kNN
acc_raw = KNeighborsClassifier(n_neighbors=5).fit(X_tr, y_tr).score(X_te, y_te)

# (c) standardized kNN -- scaler fitted on TRAIN only
scaler = StandardScaler().fit(X_tr)
Z_tr, Z_te = scaler.transform(X_tr), scaler.transform(X_te)
acc_scaled = KNeighborsClassifier(n_neighbors=5).fit(Z_tr, y_tr).score(Z_te, y_te)

print(f"acc_raw    = {acc_raw:.3f}")     # 0.55  -- pressure's scale drowns the signal
print(f"acc_scaled = {acc_scaled:.3f}")  # 0.90  -- the informative features can vote

Raw accuracy 0.55 — barely better than coin-flipping, because the
pressure column (spread ~1400, *carrying no class signal*) dominates
every distance.
Standardized: 0.90.
Same data, same model, same $k$; the only change is giving every
feature the same currency.
That before/after is the single most important experiment in this
unit.

### Checkpoint 7

1. In the worked example, *why* is `acc_raw` close to 0.5?
   Name the feature responsible and what property makes it poisonous
   (two things: its spread, and its signal content).
2. Rewrite (c)'s scaling using only NumPy (no `StandardScaler`),
   preserving the train-statistics discipline.
3. The zero-points clause bans "fitting the scaler on any test row".
   Which single line of a wrong solution would violate it, and what
   would that line look like?

## 8. Common Pitfalls II

**Pitfall 1 — fitting the scaler on all rows (train + test).**
The numbers barely move, which is exactly why the bug survives code
review — but the test rows have now influenced the classifier's
coordinates, and the test accuracy is no longer an honest
generalization estimate:

In [ ]:
leaky_scaler = StandardScaler().fit(X_shuf)          # BROKEN: sees all 60 rows
honest_scaler = StandardScaler().fit(X_tr)           # train rows only

print("leaky  mean:", leaky_scaler.mean_.round(2))
print("honest mean:", honest_scaler.mean_.round(2))  # close -- but not equal
print("the difference is small; the PROTOCOL violation is not.")

Fix: the scaler is part of the classifier — everything it learns must
come from training rows.
Session 3's pipelines make this structurally impossible to get wrong.

**Pitfall 2 — scaling the training set but not the query.**
Half-applied standardization is worse than none: the query is now in
different units from every training row, and distances are
meaningless:

In [ ]:
q_machine = X_te[0]                          # a raw test row
Z_tr_only = honest_scaler.transform(X_tr)

d2_wrong = ((Z_tr_only - q_machine) ** 2).sum(axis=1)     # BROKEN: raw q vs scaled X
d2_right = ((Z_tr_only - honest_scaler.transform(q_machine[None, :])) ** 2).sum(axis=1)
print("nearest neighbor, mismatched units :", np.argmin(d2_wrong),
      " (distance dominated by pressure ~ 1e5 in RAW units)")
print("nearest neighbor, consistent units :", np.argmin(d2_right))

Fix: one rule, no exceptions — *whatever transformation the training
set got, every query gets, with the training set's statistics.*

**Pitfall 3 — even k and the silent tie.**
With $k = 4$ and a 2–2 vote there is no majority; some tie-break fires
(our mean-rule calls `> 0.5` False and answers 0; sklearn has its own
convention).
Predictions become an artifact of tie-breaking, not of evidence:

In [ ]:
X_tie = np.array([[0., 1.], [0., -1.], [2., 1.], [2., -1.]])
y_tie = np.array([0, 0, 1, 1])
q_mid = np.array([[1.0, 0.0]])               # exactly between the pairs

for k in (2, 4):
    print(f"k={k}: our rule -> {knn_predict(X_tie, y_tie, q_mid, k)[0]}",
          f"| sklearn -> {KNeighborsClassifier(n_neighbors=k).fit(X_tie, y_tie).predict(q_mid)[0]}",
          " (a 50/50 vote either way)")
print("k=3 has no tie:", knn_predict(X_tie, y_tie, q_mid, 3)[0])

Fix: odd $k$ for binary problems; more generally, avoid $k$ divisible
by the number of classes.

**Pitfall 4 — reporting training accuracy.**
Section 4 showed $k = 1$ scores 1.0 on its own training set by
construction.
Any reported number must come from rows the model never saw
(C1's discipline; Session 3 does it properly):

In [ ]:
model = KNeighborsClassifier(n_neighbors=1).fit(Z_tr, y_tr)
print("k=1 'accuracy' on training rows:", model.score(Z_tr, y_tr))       # 1.0, always
print("k=1 accuracy on unseen rows    :", round(model.score(honest_scaler.transform(X_te), y_te), 3))

### Checkpoint 8

1. A teammate standardizes `X_all` (all 60 rows) once, *then* splits
   into train and test, and reports test accuracy.
   Which pitfall is this, and what makes the reported number
   untrustworthy even though the code runs clean?
2. Why is scaling only the training set (Pitfall 2) typically *worse*
   than scaling nothing at all?
3. For a 3-class problem, name a bad choice of $k$ from the tie
   standpoint and a better nearby choice.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Points that are close in feature space tend to share a label.
   Reasonable: bean species from size/mass measurements (similar beans
   are the same variety).
   Dubious: e.g. classifying whether a number is prime from its value
   — closeness in value says nothing about the label.
2. It stores the entire training set `(X_train, y_train)`; the work
   (distances, sorting, voting) happens at prediction time, per
   query.
3. `X_train (50, 2)`, `y_train (50,)`, queries `(7, 2)`, the
   distance matrix `(7, 50)`, predictions `(7,)`.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. Differences $(2, -1, -2)$: squared Euclidean $= 4 + 1 + 4 = 9$
   (distance 3); Manhattan $= 2 + 1 + 2 = 5$.
2. $t \mapsto \sqrt{t}$ is strictly increasing, so
   $d_1^2 < d_2^2 \iff d_1 < d_2$: the sorted order is identical.
3. Shape `(3, 50, 2)` (here `(m, n, d)` in general); slice `[i, j, :]`
   holds the feature-wise differences between query $i$ and training
   row $j$.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Squared distances from $(5,5)$: $(1,1)\!: 32$, $(2,2)\!: 18$,
   $(3,3)\!: 8$, $(6,5)\!: 1$, $(7,7)\!: 8$.
   $k=1$: row 3 (label 1) → **1**.
   $k=3$: dist² 1, 8, 8 → rows 3, 2, 4 (labels 1, 0, 1) → vote 2–1 →
   **1**.
2. `y_tr[nearest]` is the `(m, k)` array of the labels of each query's
   $k$ nearest training rows; with 0/1 labels the mean is the fraction
   of 1-votes, and a fraction above $\tfrac12$ means 1s hold the
   majority.
3. The mean of labels 0/1/2 is not a class (e.g. votes {0, 2} average
   to 1 — a class nobody voted for).
   Count each label's occurrences per row (`np.unique` with
   `return_counts=True`, or compare against each class) and take the
   argmax.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Each training point's distance to itself is 0, beating every other
   (distinct) point, so its own label wins the vote: perfect training
   accuracy with zero generalization content — C1's **overfitting**
   (memorization) in its purest form.
2. Every query gets the majority class (0); training accuracy is
   $25/40 = 0.625$ — the majority-baseline from C1.
3. $k = 1$: every query whose nearest neighbor is the flipped point
   changes — the boundary redraws around it.
   $k = 25$: one vote among 25 almost never flips a majority, so
   predictions barely move.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Heights become 1800, 1600, 1780 mm.
   $d^2(q, A) = 20^2 + 800^2 = 640400$;
   $d^2(q, B) = 180^2 + 200^2 = 72400$.
   The 1-NN is **still B** — income still dominates ($800^2$ vs
   $20^2$), millimeters weren't extreme enough to hand height the
   dictatorship.
   (Kilometers-vs-dollars would flip it — the point is the answer
   depends on units at all.)
2. Influence ratio ≈ $18000^2 / 12^2 = 2.25\times10^6$ — salary
   carries about two million times age's weight in $d^2$.
3. **False** — scaling *all* features by the same constant $c$ scales
   every squared distance by $c^2$, preserving the order; only
   *unequal* per-feature scaling reorders neighbors.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Heights 1.80, 1.60: $\mu = 1.7$, $\sigma = 0.1$ →
   $(1.80 - 1.7)/0.1 = +1$, $(1.60 - 1.7)/0.1 = -1$.
   In general: subtracting $\mu_j$ makes the mean 0 (expectation is
   linear); a variable minus its mean, divided by $\sigma_j$, has
   variance $\operatorname{Var}/\sigma_j^2 = 1$ (F5: scaling by $c$
   multiplies variance by $c^2$).
2. (i) A single query row has no spread — statistics of one row are
   meaningless; (ii) letting queries/test rows into $\mu, \sigma$
   leaks information the model must not have at evaluation time,
   invalidating the test.
3. $\sigma_j = 0$ → division by zero.
   A constant feature distinguishes nothing — drop it (sklearn's
   scaler quietly maps such a column to zeros).

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. `pressure_pa`: enormous spread (~1400 in raw units, dwarfing
   temperature's ~2 and vibration's ~0.7) **and no class signal** —
   the dominant term in every distance is noise, so neighbors are
   effectively random and accuracy sits near 0.5.
2. `mu, sd = X_tr.mean(axis=0), X_tr.std(axis=0)` then
   `Z_tr, Z_te = (X_tr - mu) / sd, (X_te - mu) / sd`.
3. The fit line: `scaler = StandardScaler().fit(X_shuf)` (or
   `fit(np.vstack([X_tr, X_te]))`) — any `fit` whose argument contains
   test rows.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Pitfall 1 (leaky scaling): the scaler's $\mu, \sigma$ were computed
   from rows later used for testing, so test rows influenced the
   model's coordinate system — the protocol, not the arithmetic, is
   broken, and the estimate is (mildly, unquantifiably) optimistic.
2. Scaling nothing at least keeps the query and the training rows in
   the *same* (bad) units; scaling only the training set compares
   z-scores (~unit scale) with raw values (arbitrary scale) — pure
   nonsense distances.
3. Bad: $k = 3$ (or 6, 9) can split 1–1–1 among three classes; better:
   $k = 5$ (or 7), which cannot three-way tie… though 2–2–1 remains
   possible — no $k$ removes all ties with 3 classes, so prefer $k$
   with a clear plurality most of the time and a fixed documented
   tie-break.

</details>